In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# A Parser for Arithmetic Expressions

## Representing Arithmetic Expressions

Arithmetic expressions are built from numbers and variables using the operators `+`, `-`, `*`, `/`, and `**`, where `**` denotes exponentiation.  Furthermore, functions like `exp` or `ln` can be applied to expressions.  We represent arithmetic expressions by the discriminated union `Expr`:

| Expression       | F# representation       |
|:-----------------|:------------------------|
| $42$             | `Num 42`                |
| $x$              | `Var "x"`               |
| $f + g$          | `Add (f, g)`            |
| $f - g$          | `Sub (f, g)`            |
| $f \cdot g$      | `Mul (f, g)`            |
| $f / g$          | `Div (f, g)`            |
| $f^g$            | `Pow (f, g)`            |
| $\exp(f)$        | `App ("exp", f)`        |

For example, the expression `x * exp(x)` is represented as `Mul (Var "x", App ("exp", Var "x"))`.

In [2]:
type Expr =
    | Num of int             // number
    | Var of string          // variable
    | Add of Expr * Expr     // +
    | Sub of Expr * Expr     // -
    | Mul of Expr * Expr     // *
    | Div of Expr * Expr     // /
    | Pow of Expr * Expr     // **
    | App of string * Expr   // application of a function like exp or ln

## The Tokenizer

The function `tokenize` takes one argument:
- `s` is a string that is supposed to represent an arithmetic expression.

It returns the list of tokens of `s`.  A token is either
- a number, i.e. a sequence of digits that does not start with `0`,
- a function name, i.e. a sequence of at least two lower case letters,
- a variable, i.e. a single lower case letter,
- the power operator `**`, or
- one of the arithmetical operators `+`, `-`, `*`, `/`, or a parenthesis.

Blanks and tabs are skipped.

**Implementation:** The tokens are specified by a regular expression that has one group for every kind of token.  If the first group, which matches blanks and tabs, has matched, the match is discarded.  Note that the group for the power operator `**` has to come before the group for the operator `*`.

In [3]:
open System.Text.RegularExpressions

let tokenize (s: string) : string list =
    let lexSpec = """ ([\ \t]+)      |  # blanks and tabs
                      ([1-9][0-9]*)  |  # number
                      ([a-z]{2,})    |  # function name
                      ([a-z])        |  # identifier
                      (\*\*)         |  # power operator
                      ([+\-*/()])       # arithmetical operators
                  """
    [ for m in Regex.Matches(s, lexSpec, RegexOptions.IgnorePatternWhitespace) do
        if not m.Groups[1].Success then
            m.Value ]

In [4]:
tokenize "x * y + 2 * x ** 3 - sin(x)"

["x"; "*"; "y"; "+"; "2"; "*"; "x"; "**"; "3"; "-"; "sin"; "("; "x"; ")"]

The functions `isIdentifier`, `isNumber`, and `isFunction` take one argument:
- `s` is a token.

They check whether `s` is a variable, a number, or a function name, respectively.

In [5]:
let isIdentifier (s: string) : bool = Regex.IsMatch(s, "^[a-z]$")
let isNumber     (s: string) : bool = Regex.IsMatch(s, "^[1-9][0-9]*$")
let isFunction   (s: string) : bool = Regex.IsMatch(s, "^[a-z]{2,}$")

The function `precedence` takes one argument:
- `op` is an operator or a function name.

It returns the precedence of `op`.  Function application binds strongest.

In [6]:
let precedence (op: string) : int =
    match op with
    | "+" | "-" -> 1
    | "*" | "/" -> 2
    | "**"      -> 3
    | _         -> 4

The function `precedenceOp` takes one argument:
- `e` is an expression.

It returns the precedence of the outermost operator of `e`.  Numbers, variables, and function applications have the highest precedence $4$.

In [7]:
let precedenceOp (e: Expr) : int =
    match e with
    | Add _ | Sub _ -> 1
    | Mul _ | Div _ -> 2
    | Pow _         -> 3
    | _             -> 4

## Converting Expressions into Strings

The function `toString` takes one argument:
- `e` is an expression.

It returns a readable string representing `e`.  Parentheses are only used where they are necessary:
- In `f - g`, the right argument `g` has to be parenthesized if it is a sum or a difference.
- In `f * g`, both arguments have to be parenthesized if they are sums or differences.
- In `f / g`, the left argument has to be parenthesized if it is a sum or difference, the right argument has to be parenthesized if it is a sum, difference, product, or quotient.
- In `f ** g`, the left argument has to be parenthesized unless it is a number, a variable, or a function application, while the right argument has to be parenthesized if it is a sum, difference, product, or quotient.

The local function `paren` takes a condition and an expression.  It converts the expression into a string and encloses it in parentheses if the condition is true.

In [8]:
let rec toString (e: Expr) : string =
    let paren (condition: bool) (e: Expr) : string =
        if condition then "(" + toString e + ")" else toString e
    match e with
    | Num n      -> string n
    | Var x      -> x
    | App (f, a) -> f + "(" + toString a + ")"
    | Add (f, g) -> toString f + " + " + toString g
    | Sub (f, g) -> toString f + " - " + paren (precedenceOp g = 1) g
    | Mul (f, g) -> paren (precedenceOp f = 1) f + "*" + paren (precedenceOp g = 1) g
    | Div (f, g) -> paren (precedenceOp f = 1) f + "/" + paren (precedenceOp g <= 2) g
    | Pow (f, g) -> paren (precedenceOp f <= 3) f + "**" + paren (precedenceOp g <= 2) g

## The Shunting Yard Algorithm

The parser uses the *shunting yard algorithm*, which has already been discussed in the notebook `Propositional-Logic-Parser.ipynb`.

The function `evalBefore` takes two arguments:
- `stackOp` is the operator on top of the operator stack,
- `nextOp` is the next operator from the input.

It returns `true` if `stackOp` has to be evaluated before `nextOp` is processed.  The operators `+`, `-`, `*`, and `/` are left associative, while `**` is right associative.

In [9]:
let evalBefore (stackOp: string) (nextOp: string) : bool =
    if stackOp = "(" then
        false
    elif precedence stackOp > precedence nextOp then
        true
    elif precedence stackOp = precedence nextOp then
        if stackOp = nextOp then
            List.contains stackOp [ "+"; "-"; "*"; "/" ]   // left associative
        else
            true
    else
        false

The function `apply` takes two arguments:
- `op` is an operator or a function name,
- `arguments` is the argument stack, represented as a list where the top of the stack is the head of the list.

It applies `op` to the topmost arguments on the stack and pushes the result.  A function takes one argument, an operator takes two arguments.  The new stack is returned.

In [10]:
let apply (op: string) (arguments: Expr list) : Expr list =
    match arguments with
    | arg :: rest when isFunction op -> App (op, arg) :: rest
    | rhs :: lhs :: rest ->
        let e = match op with
                | "+"  -> Add (lhs, rhs)
                | "-"  -> Sub (lhs, rhs)
                | "*"  -> Mul (lhs, rhs)
                | "/"  -> Div (lhs, rhs)
                | "**" -> Pow (lhs, rhs)
                | _    -> failwithf "unknown operator %s" op
        e :: rest
    | _ -> failwithf "syntax error: missing argument for %s" op

The function `parse` takes one argument:
- `s` is a string representing an arithmetic expression.

It returns the expression represented by `s`.  The implementation is the same as the implementation of the parser for propositional logic:  The tokens are processed from left to right.  Numbers and variables are pushed onto the argument stack.  For operators and function names, we compare the precedence of the operator on top of the operator stack with the precedence of the next operator to decide whether the operator on the stack has to be evaluated first.  A function name is treated like an operator that has the highest precedence.

In [11]:
let parse (s: string) : Expr =
    let mutable tokens    = tokenize s
    let mutable operators : string list = []
    let mutable arguments : Expr list   = []
    while not tokens.IsEmpty do
        let nextOp = tokens.Head
        tokens <- tokens.Tail
        if isIdentifier nextOp then
            arguments <- Var nextOp :: arguments
        elif isNumber nextOp then
            arguments <- Num (int nextOp) :: arguments
        elif operators.IsEmpty || nextOp = "(" then
            operators <- nextOp :: operators
        else
            let stackOp = operators.Head
            if stackOp = "(" && nextOp = ")" then
                operators <- operators.Tail
            elif nextOp = ")" || evalBefore stackOp nextOp then
                operators <- operators.Tail
                arguments <- apply stackOp arguments
                tokens    <- nextOp :: tokens
            else
                operators <- nextOp :: operators
    for op in operators do
        arguments <- apply op arguments
    match arguments with
    | [ e ] -> e
    | _     -> failwithf "could not parse %s" s

## Testing

The function `testParser` takes one argument:
- `s` is a string representing an arithmetic expression.

It parses `s` and prints the resulting expression both as a value of type `Expr` and as a string.

In [12]:
let testParser (s: string) : unit =
    printfn "parsing %s" s
    let t = parse s
    print t
    printfn "%s\n" (toString t)

In [13]:
testParser "x ** y"
testParser "x ** y ** z"
testParser "(x ** y) ** z"
testParser "x * y + 2 * x ** 3"
testParser "sin(asin(x))"
testParser "(x - y) - (a - b)"

parsing 

x ** y

Pow (Var "x", Var "y")

x**y

parsing 

...

sin(asin(x))

parsing 

(x - y) - (a - b)

Sub (Sub (Var "x", Var "y"), Sub (Var "a", Var "b"))

x - y - (a - b)